# Studying the importance of qubit placement on BB schedule metrics

In [ ]:
import os
try:
    path_initialized
except NameError:
    path_initialized = True
    os.chdir('..')

import numpy as np
from sympy.abc import x, y

from qldpc import codes
from qldpc.objects import Pauli
import networkx as nx
import matplotlib.pyplot as plt

import src.device as device
import src.plotting as plotter
from src.RotatedSurfaceCode import RotatedSurfaceCode
from src.HGPCode import HGPCode
from src.GBCode import GBCode
from src.QECCode import TestCode

In [ ]:
l,m = 12,6
orders = {x: l, y: m}
poly_a = y+y**2+x**3
poly_b = y**3+x+x**2
code = codes.BBCode(orders, poly_a, poly_b)

print(code)
print()
print("number of logical qubits:", code.dimension)

# find an upper bound to the code distance with 100 Monte Carlo trials
# print("code distance: <=", code.get_distance_bound(num_trials=100))

In [ ]:
code.get_logical_ops(Pauli.X).shape

In [ ]:
code_bb = GBCode(12, 6, ([3], [1, 2]), ([1, 2], [3]))

In [ ]:
code_bb.num_data

In [ ]:
code_bb.compute_logical_operators()[0].shape

In [ ]:
np.count_nonzero(code_bb.compute_logical_operators()[0], axis=0)

In [ ]:
code_bb = GBCode(12, 6, ([3], [1, 2]), ([1, 2], [3]))

buffer = 3
xmax,ymax = 0,0
data_coords = dict()
for i in code_bb.data_indices:
    x,y = code_bb.qubit_coords[i]
    data_coords[i] = (x+buffer, y+buffer)
    xmax = max(xmax, x)
    ymax = max(ymax, y)

dev = device.UnitCellDevice(xmax+2*buffer, ymax+2*buffer, device.default_hwp)

sched_bb = dev.compile_QEC_schedule(
    code_bb,
    data_coords,
    [],
    rounds=12,
    use_highways=True,
    refocus_shuttle_noise=False,
    optimize_ancilla_start=True,
    separate_X_Z=True
)